# Merge Wheel Wear and Hunting Results

Join the wheel-wear workbook to the axle and truck simulation results. See
[INSTALL.md](../INSTALL.md) for the input layout. This notebook writes new CSVs
to `outputs/data/`. The other notebooks can use the merged CSV supplied in
`data/`.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository; see INSTALL.md.")
sys.path.insert(0, str(ROOT / "src"))
from project_paths import data_directory, find_project_root, output_directory, required_file, wheel_wear_file
ROOT = find_project_root(ROOT)
DATA = data_directory(ROOT)

import pandas as pd

sorted(p.name for p in DATA.iterdir())

In [ ]:
wheel_wear = pd.read_excel(wheel_wear_file(DATA))

wheel_wear.head()

In [ ]:
wheel_wear.shape

In [ ]:
eq_file = pd.ExcelFile(required_file(DATA / "Eq_Con_Fit_Results.xlsm"))

eq_file.sheet_names

In [ ]:
eq_con = pd.read_excel(
    required_file(DATA / "Eq_Con_Fit_Results.xlsm"),
    sheet_name="Eq_Con_Fit_Results"
)

eq_con.head()

In [ ]:
eq_con.shape

In [ ]:
# Remove the 7 KRWHL rows since they do not have matching Car/Axle data in the wheel wear file
eq_con_cars = eq_con[
    eq_con["Filename"].str.startswith("Car", na=False)
].copy()


eq_con_cars.shape

In [ ]:
# Extract the car number, axle number, and rail profile number from each filename
eq_con_cars["Car"] = eq_con_cars["Filename"].str.extract(r"Car(\d+)").astype(int)
eq_con_cars["Axle"] = eq_con_cars["Filename"].str.extract(r"Axle(\d+)").astype(int)
eq_con_cars["Rail_Profile"] = eq_con_cars["Filename"].str.extract(r"RProfile\s*(\d+)").astype(int)

eq_con_cars[["Filename", "Car", "Axle", "Rail_Profile"]].head(10)

In [ ]:
# Extract the car number, axle number, and left/right wheel side from each wheel wear filename
wheel_wear["Car"] = wheel_wear["Filename"].str.extract(r"Car(\d+)").astype(int)
wheel_wear["Axle"] = wheel_wear["Filename"].str.extract(r"Axle(\d+)").astype(int)
wheel_wear["Side"] = wheel_wear["Filename"].str.extract(r"Axle\d+([LR])")

wheel_wear[["Filename", "Car", "Axle", "Side"]].head(10)

In [ ]:
wheel_wear.columns.tolist()

In [ ]:
# Remove all blank Excel columns labeled "Unnamed" so only the actual wheel wear data remains
wheel_wear = wheel_wear.loc[
    :, ~wheel_wear.columns.str.startswith("Unnamed:")
]

wheel_wear.columns.tolist()

In [ ]:
wheel_wear.dtypes

In [ ]:
# Remove identifier/text columns so only the numeric wheel measurements remain
wheel_measurements = wheel_wear.drop(
    columns=[
        "Filename",
        "Axle Number",
        "Wheel Side",
        "Hollow wear"
    ]
)

# Reshape the data so each Car/Axle has separate left and right wheel measurements
wear_axle = wheel_measurements.pivot(
    index=["Car", "Axle"],
    columns="Side"
)

# Rename the left and right measurement columns
new_columns = []

for measurement, side in wear_axle.columns:
    new_name = measurement + "_" + side
    new_columns.append(new_name)

wear_axle.columns = new_columns

# Turn Car and Axle back into regular columns
wear_axle = wear_axle.reset_index()

wear_axle.shape

In [ ]:
print(wear_axle.shape)

wear_axle.columns.tolist()

In [ ]:
# Merge the wheel measurements into the Eq Con data by matching the same Car and Axle
merged_data = eq_con_cars.merge(
    wear_axle,
    on=["Car", "Axle"],
    how="left",
    validate="many_to_one"
)

merged_data.shape

In [ ]:
# Check that every Eq Con row received matching wheel wear data
missing_rows = merged_data["qR_L"].isna().sum()

print("Rows missing wheel wear data:", missing_rows)

In [ ]:
# List the four hunting-speed columns that can be used as our outcome variables
hunting_columns = [
    "Empty Truck from Acc HS [mph]",
    "Empty Axle from Lat Disp HS [mph]",
    "Loaded Truck from Acc HS [mph]",
    "Loaded Axle from Lat Disp HS [mph]"
]

merged_data[hunting_columns].dtypes

In [ ]:
# Classify each row as hunting if the empty-car hunting speed is below 80 mph
merged_data["Empty_Truck_Hunted"] = merged_data["Empty Truck from Acc HS [mph]"] < 80
merged_data["Empty_Axle_Hunted"] = merged_data["Empty Axle from Lat Disp HS [mph]"] < 80

# Compare class counts for the two hunting criteria.
print(merged_data["Empty_Truck_Hunted"].value_counts())
print(merged_data["Empty_Axle_Hunted"].value_counts())

In [ ]:
# Save the axle merge before adding truck results.
merged_data.to_csv(output_directory(ROOT, "data") / "merged_wheel_hunting_data.csv", index=False)

merged_data.columns.tolist()

In [ ]:
# Add the truck number so each axle can be matched with the correct truck-level data
merged_data["Truck"] = 1

merged_data.loc[merged_data["Axle"] >= 3, "Truck"] = 2

merged_data["Truck"].value_counts()

In [ ]:
# Load the truck-level results from the second sheet of the Eq Con Excel file
truck_eq = pd.read_excel(
    required_file(DATA / "Eq_Con_Fit_Results.xlsm"),
    sheet_name="Truck_Eq_Con_Res"
)

truck_eq.shape

# Keep only the truck rows associated with Cars 1–14 and remove the KRWHL rows
truck_cars = truck_eq[
    truck_eq["Filename"].str.startswith("Car", na=False)
].copy()

truck_cars.shape

In [ ]:
# Extract the car, truck, and rail profile numbers so the truck data can match our existing merged dataset
truck_cars["Car"] = truck_cars["Filename"].str.extract(r"Car(\d+)").astype(int)
truck_cars["Truck"] = truck_cars["Filename"].str.extract(r"Truck(\d+)").astype(int)
truck_cars["Rail_Profile"] = truck_cars["Filename"].str.extract(r"RProfile\s*(\d+)").astype(int)

truck_cars[["Filename", "Car", "Truck", "Rail_Profile"]].head(10)

In [ ]:
# Merge the truck-level results into the existing dataset using Car, Truck, and Rail Profile
final_data = merged_data.merge(
    truck_cars,
    on=["Car", "Truck", "Rail_Profile"],
    how="left",
    suffixes=("", "_Truck"),
    validate="many_to_one"
)

final_data.shape

In [ ]:
# Check that every row received matching truck-level data
missing_truck = final_data["Min Eq Con 3mm"].isna().sum()

print("Rows missing truck data:", missing_truck)

In [ ]:
# Save the final merged dataset under outputs/data/
final_data.to_csv(output_directory(ROOT, "data") / "final_merged_data.csv", index=False)

## What the merge contains

The 112 measured wheels become 56 axle rows with separate left and right
measurements. Joining the seven rail conditions gives 392 axle/rail rows.
We exclude the unworn KRWHL reference rows because they have no measured-car
wheel-wear matches.

Truck 1 contains axles 1–2 and Truck 2 contains axles 3–4. We join truck results
by car, truck and rail profile; overlapping truck column names receive the
`_Truck` suffix. The final table has 392 rows and 94 columns, with no unmatched
wheel-wear or truck records in the supplied data.

`Empty_Truck_Hunted` uses the empty-truck acceleration criterion, and
`Empty_Axle_Hunted` uses axle displacement. Values below 80 mph count as hunting;
80 means no onset was recorded below the simulation limit. These are separate
outcomes, and two axle rows share each truck result.